# Custom atlases

Lacuna ships with several bundled atlases (`lacuna info atlases`), but you can bring your own. This guide shows how to use custom parcellation atlases with Regional Damage and Structural Network Mapping.

**What you'll learn**:

- Prepare a custom atlas and labels file
- Run Regional Damage with the XTRACT white matter tract atlas
- Run Structural Network Mapping with the Yeo 2011 functional network atlas

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/m-petersen/lacuna/blob/main/docs/how-to/custom-atlases.ipynb)

## Colab

Note: Colab provides limited computational resources. While these tutorials are designed to operate within those constraints, some Lacuna functionality cannot be fully demonstrated in this environment and requires access to higher-performance computing infrastructure.

Ignore this if you run this notebook locally.

In [2]:
# --- Conda setup for Google Colab ---
# Lacuna's structural network mapping capabilities rely on MRtrix3.
# However, Colab does not include conda by default, which is required for MRtrix3 installation.
# We use condacolab to replace Colab's Python with a Mambaforge-based environment.
#
# IMPORTANT: Running this cell will RESTART the kernel automatically.
# After the restart, SKIP this cell and continue from the next cell onward.
# (Do NOT re-run this cell after the restart.)

import os

if os.environ.get("CONDA_PREFIX") is None:

    !pip install -q condacolab==0.1.9
    import condacolab
    condacolab.install()  # This restarts the kernel

In [3]:
# Run this after the kernel restart.
import sys

if 'google.colab' in sys.modules:

    import condacolab
    import subprocess

    condacolab.check()

    conda_paths = subprocess.check_output(
        "python -c 'import sys; print(\"\\n\".join(sys.path))'",
        shell=True, text=True
    ).split('\n')

    for path in conda_paths:
        if path and path not in sys.path:
            sys.path.append(path)

    print("Colab environment detected: condacolab initialized.")

## Setup

In [ ]:
# Install Lacuna from GitHub
!pip install git+https://github.com/m-petersen/lacuna

# Install MRtrix3 via conda (needed for SNM section)
!conda install -c conda-forge -c MRtrix3 mrtrix3 libstdcxx-ng -y

In [4]:
!lacuna tutorial /tmp/tutorial_bids --force


Setting up tutorial data at: /tmp/tutorial_bids
✓ Tutorial data copied to: /tmp/tutorial_bids

The tutorial dataset includes:
  - 3 synthetic subjects (sub-01, sub-02, sub-03)
  - Binary lesion masks in MNI152NLin6Asym space
  - BIDS-compliant structure


### How Lacuna handles custom atlases

All `lacuna run` analyses accept the `--custom-parcellation` flag:

```
--custom-parcellation <NIFTI> <LABELS> <SPACE>
```

| Argument | Description |
|----------|-------------|
| `NIFTI`  | Path to the parcellation NIfTI file (3D integer-labeled or 4D probabilistic) |
| `LABELS` | Path to a text file mapping region IDs to names |
| `SPACE`  | Coordinate space of the atlas (e.g., `MNI152NLin6Asym`) |

The flag can be specified multiple times and combined with `--parcel-atlases`.

### Labels file format

The labels file is plain text with one region per line. Each line contains a numeric region ID and the region name, separated by a space:

```text
1 Left-Precentral
2 Left-Postcentral
3 Left-SMA
```

Lines starting with `#` are treated as comments and ignored.

---

## Regional Damage with the XTRACT tract atlas

The [XTRACT](https://github.com/SPMIC-UoN/XTRACT_atlases) atlas provides maps of major white matter tracts. Using it with Regional Damage tells you which tracts overlap with a lesion.

### Download the atlas

In [5]:
!wget -q -O /tmp/xtract-maxprob5-1mm.nii.gz \
    "https://github.com/SPMIC-UoN/XTRACT_atlases/raw/master/fsleyes_atlases/XTRACT/xtract-tract-atlases-maxprob5-1mm.nii.gz"

!wget -q -O /tmp/xtract-maxprob5-1mm.xml \
    "https://github.com/SPMIC-UoN/XTRACT_atlases/raw/master/fsleyes_atlases/XTRACT.xml"

print("Downloaded XTRACT atlas and label file.")

Downloaded XTRACT atlas and label file.


### 2. Create a labels file

The XTRACT atlas ships with an XML label file. Convert it to a plain text labels file that Lacuna can read.

In [ ]:
import xml.etree.ElementTree as ET

tree = ET.parse("/tmp/xtract-maxprob5-1mm.xml")
root = tree.getroot()

with open("/tmp/xtract_labels.txt", "w") as f:
    for label in root.iter("label"):
        # XTRACT XML uses 0-based indices; Lacuna labels are 1-based (0 is background)
        index = int(label.get("index")) + 1
        name = label.text.strip()
        f.write(f"{index} {name}\n")

!cat /tmp/xtract_labels.txt

1 Anterior Commissure
2 Arcuate Fasciculus L
3 Arcuate Fasciculus R
4 Acoustic Radiation L
5 Acoustic Radiation R
6 Anterior Thalamic Radiation L
7 Anterior Thalamic Radiation R
8 Cingulum subsection: Dorsal L
9 Cingulum subsection: Dorsal R
10 Cingulum subsection: Peri-genual L
11 Cingulum subsection: Peri-genual R
12 Cingulum subsection: Temporal L
13 Cingulum subsection: Temporal R
14 Corticospinal Tract L
15 Corticospinal Tract R
16 Frontal Aslant Tract L
17 Frontal Aslant Tract R
18 Forceps Major
19 Forceps Minor
20 Fornix L
21 Fornix R
22 Inferior Fronto-Occipital Fasciculus L
23 Inferior Fronto-Occipital Fasciculus R
24 Inferior Longitudinal Fasciculus L
25 Inferior Longitudinal Fasciculus R
26 Middle Cerebellar Peduncle
27 Middle Longitudinal Fasciculus L
28 Middle Longitudinal Fasciculus R
29 Optic Radiation L
30 Optic Radiation R
31 Superior Longitudinal Fasciculus 1 L
32 Superior Longitudinal Fasciculus 1 R
33 Superior Longitudinal Fasciculus 2 L
34 Superior Longitudinal Fas

### Run analysis

The XTRACT atlas is in MNI152NLin6Asym space (FSL's standard MNI space).

In [62]:
!lacuna run rd \
    /tmp/tutorial_bids/ \
    /tmp/outputs_rd_xtract/ \
    --participant-label 01 \
    --mask-space MNI152NLin6Asym \
    --custom-parcellation \
        /tmp/xtract-maxprob5-1mm.nii.gz \
        /tmp/xtract_labels.txt \
        MNI152NLin2009cAsym \
    --verbose

2026-03-25 14:12:16 - lacuna.cli.main - INFO - Lacuna CLI starting
2026-03-25 14:12:16 - lacuna.cli.main - INFO - Input: /tmp/tutorial_bids
2026-03-25 14:12:16 - lacuna.cli.main - INFO - Output directory: /tmp/outputs_rd_xtract
2026-03-25 14:12:16 - lacuna.cli.main - INFO - Analysis: rd
2026-03-25 14:12:16 - lacuna.cli.main - INFO - Registering custom parcellation: xtract-maxprob5-1mm (space=MNI152NLin2009cAsym, res=1mm)
2026-03-25 14:12:16 - lacuna.cli.main - INFO - Running analysis: RegionalDamage
2026-03-25 14:12:16 - lacuna.cli.main - INFO - 
2026-03-25 14:12:16 - lacuna.cli.main - INFO - ============================================================
2026-03-25 14:12:16 - lacuna.cli.main - INFO - DISCOVERY SUMMARY
2026-03-25 14:12:16 - lacuna.cli.main - INFO - ============================================================
2026-03-25 14:12:16 - lacuna.cli.main - INFO -   Total mask images: 1
2026-03-25 14:12:16 - lacuna.cli.main - INFO -   Unique subjects:   1
2026-03-25 14:12:16 - lacu

In [8]:
!ls /tmp/outputs_rd_xtract/sub-01/ses-01/anat/

sub-01_ses-01_desc-provenance.json
sub-01_ses-01_label-acuteinfarct_atlas-xtract-maxprob5-1mm_source-regionaldamage_desc-damagebin_parcelstats.json
sub-01_ses-01_label-acuteinfarct_atlas-xtract-maxprob5-1mm_source-regionaldamage_desc-damagebin_parcelstats.tsv
sub-01_ses-01_label-acuteinfarct_atlas-xtract-maxprob5-1mm_source-regionaldamage_desc-damagepct_parcelstats.json
sub-01_ses-01_label-acuteinfarct_atlas-xtract-maxprob5-1mm_source-regionaldamage_desc-damagepct_parcelstats.tsv
sub-01_ses-01_space-MNI152NLin6Asym_label-acuteinfarct_mask.nii.gz


### Inspect the results

The output shows what percentage of each white matter tract is overlapped by the lesion.

In [9]:
import pandas as pd
from pathlib import Path

output_dir = Path("/tmp/outputs_rd_xtract/sub-01/ses-01/anat/")
tsv_files = list(output_dir.glob("*damagepct*parcelstats.tsv"))

df = pd.read_csv(tsv_files[0], sep="\t")
df.sort_values(by="value", ascending=False).head(10)

,region,value
22,Inferior Fronto-Occipital Fasciculus R,5.251906
17,Forceps Major,4.460023
2,Arcuate Fasciculus R,0.000000
3,Acoustic Radiation L,0.000000
4,Acoustic Radiation R,0.000000
5,Anterior Thalamic Radiation L,0.000000
6,Anterior Thalamic Radiation R,0.000000
7,Cingulum subsection: Dorsal L,0.000000
0,Anterior Commissure,0.000000
1,Arcuate Fasciculus L,0.000000


### Visualize lesion and tract overlap

Overlay the lesion with the most affected tract (Inferior Fronto-Occipital Fasciculus R, label 23 in the XTRACT atlas).

In [ ]:
import nibabel as nib
import numpy as np
from nilearn import plotting

xtract_img = nib.load("/tmp/xtract-maxprob5-1mm.nii.gz")
xtract_data = xtract_img.get_fdata()

# Extract the Inferior Fronto-Occipital Fasciculus R (label 23)
xtract_ifof = (xtract_data == 23).astype(np.int8)
roi_img = nib.Nifti1Image(xtract_ifof, affine=xtract_img.affine, header=xtract_img.header)

lesion_img = nib.load("/tmp/tutorial_bids/sub-01/ses-01/anat/sub-01_ses-01_space-MNI152NLin6Asym_label-acuteinfarct_mask.nii.gz")

plot = plotting.plot_roi(roi_img,
                         radiological=True,
                         title="Inferior Fronto-Occipital Fasciculus R",
                         display_mode="z",
                         draw_cross=False,
                         colorbar=False)

plot.add_overlay(lesion_img, cmap='Reds_r', transparency=0.5)

You can also combine custom and bundled atlases in a single run:

In [ ]:
!lacuna run rd \
    /tmp/tutorial_bids/ \
    /tmp/outputs_rd_combined/ \
    --participant-label 01 \
    --mask-space MNI152NLin6Asym \
    --parcel-atlases Schaefer2018_100Parcels7Networks \
    --custom-parcellation \
        /tmp/xtract-maxprob5-1mm.nii.gz \
        /tmp/xtract_labels.txt \
        MNI152NLin6Asym \
    --verbose

---

## Structural Network Mapping with the Yeo 2011 atlas

The [Yeo 2011](https://doi.org/10.1152/jn.00338.2011) atlas parcellates the cortex into functional networks. Using it with SNM computes structural disconnection and disconnectivity matrices for these networks.

### 1. Fetch the atlas with nilearn

We use the 7-network thick cortical model and save it as a standalone NIfTI. The Yeo atlas is in MNI152NLin6Asym space.

In [ ]:
import nibabel as nib
from nilearn.datasets import fetch_atlas_yeo_2011

yeo = fetch_atlas_yeo_2011()

# Save the 7-network thick cortical model
img = nib.load(yeo["thick_7"])
nib.save(img, "/tmp/yeo_7networks.nii.gz")

print(f"Yeo atlas shape: {img.shape}, voxel size: {img.header.get_zooms()[:3]}")

### 2. Create a labels file

In [ ]:
labels = {
    1: "Visual",
    2: "Somatomotor",
    3: "Dorsal-Attention",
    4: "Ventral-Attention",
    5: "Limbic",
    6: "Frontoparietal",
    7: "Default",
}

with open("/tmp/yeo_7networks_labels.txt", "w") as f:
    for idx, name in labels.items():
        f.write(f"{idx} {name}\n")

!cat /tmp/yeo_7networks_labels.txt

### 3. Fetch a tractogram

SNM requires a structural connectome. We use the lightweight HCP1065 tractogram for this tutorial.

In [ ]:
!lacuna fetch hcp1065 \
    --output-dir /tmp/hcp1065_data

### 4. Run Structural Network Mapping

In [ ]:
!lacuna run snm \
    /tmp/tutorial_bids/ \
    /tmp/outputs_snm_yeo/ \
    --connectome-path /tmp/hcp1065_data/hcp1065.tck \
    --participant-label 01 \
    --mask-space MNI152NLin6Asym \
    --custom-parcellation \
        /tmp/yeo_7networks.nii.gz \
        /tmp/yeo_7networks_labels.txt \
        MNI152NLin6Asym \
    --compute-disconnectivity-matrix \
    --compute-roi-disconnection \
    --verbose

In [ ]:
!ls /tmp/outputs_snm_yeo/sub-01/ses-01/anat/

### 5. Inspect the results

View the ROI-level disconnection values for the 7 Yeo networks.

In [ ]:
output_dir = Path("/tmp/outputs_snm_yeo/sub-01/ses-01/anat/")
tsv_files = list(output_dir.glob("*roidisconnection*parcelstats.tsv"))

df = pd.read_csv(tsv_files[0], sep="\t")
df.sort_values(by="value", ascending=False)

View the 7x7 disconnectivity matrix.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

matrix_files = list(output_dir.glob("*disconnectivitypercent*connmatrix.tsv"))
mat = pd.read_csv(matrix_files[0], sep="\t", header=0, index_col=0)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(mat.values, cmap="Reds", vmin=0)
ax.set_xticks(range(len(mat.columns)))
ax.set_xticklabels(mat.columns, rotation=45, ha="right")
ax.set_yticks(range(len(mat.index)))
ax.set_yticklabels(mat.index)
plt.colorbar(im, label="Disconnectivity (%)")
ax.set_title("Yeo 7-network disconnectivity matrix")
plt.tight_layout()
plt.show()

---

## Space handling

Lacuna automatically transforms the custom atlas to match the input mask space if they differ. For example, if your masks are in `MNI152NLin2009cAsym` but the atlas is in `MNI152NLin6Asym`, the atlas will be warped to match. This requires the corresponding TemplateFlow transform files, which are downloaded automatically.